In [2]:
import torch
from transformers import AutoTokenizer, AutoConfig
import gc
import os
from awq import AutoAWQForCausalLM

In [3]:
# Configure environment
os.environ["TOKENIZERS_PARALLELISM"] = "false"
base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print("Checking PyTorch and CUDA versions...")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name()}")
     

Checking PyTorch and CUDA versions...
PyTorch version: 2.13.0+cu130
CUDA available: True
CUDA version: 13.0
GPU: NVIDIA GB10


In [4]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

45

### Loading tokenizer

In [ ]:
tok = AutoTokenizer.from_pretrained(
    base_model,
    use_fast=True, # use the fast Rust-based tokenizer (quicker than the Python one)
    trust_remote_code=True # allow running the model repo's own custom tokenizer code
)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

### Loading model

In [8]:
mdl = AutoAWQForCausalLM.from_pretrained(
    base_model, 
    low_cpu_mem_usage=True,  # load weights in a memory-frugal way (avoids a big RAM spike)
    use_cache=False, # don't keep the KV-cache; saves memory (fine for training/eval)
    trust_remote_code=True, # allow the model repo's custom code (same as tokenizer)
    torch_dtype=torch.float16, # store activations in 16-bit floats — half the memory of float32
    device_map={"": 0} if torch.cuda.is_available() else "cpu" 
)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [9]:
mdl.eval()

LlamaAWQForCausalLM(
  (model): LlamaForCausalLM(
    (model): LlamaModel(
      (embed_tokens): Embedding(32000, 2048)
      (layers): ModuleList(
        (0-21): 22 x LlamaDecoderLayer(
          (self_attn): LlamaAttention(
            (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
            (k_proj): Linear(in_features=2048, out_features=256, bias=False)
            (v_proj): Linear(in_features=2048, out_features=256, bias=False)
            (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          )
          (mlp): LlamaMLP(
            (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
            (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
            (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
            (act_fn): SiLUActivation()
          )
          (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
          (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)

### AWQ quantization config

In [10]:
quant_config = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM"
}

In [11]:
# Create more diverse and shorter calibration data
print("Preparing calibration data...")
calib_texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning models process data efficiently.",
    "Natural language understanding is advancing rapidly.",
    "Deep neural networks learn complex patterns.",
    "Artificial intelligence transforms technology.",
    "Computer vision recognizes objects accurately.",
    "Robotics integrates sensors and actuators.",
    "Algorithm optimization improves performance significantly.",
    "Data science extracts meaningful insights.",
    "Software engineering creates reliable systems."
] * 10  # 100 samples total

Preparing calibration data...


In [12]:
os.environ["PYTORCH_USE_SDPA"]  = "0"

In [13]:
cfg = AutoConfig.from_pretrained(base_model, trust_remote_code=True)
cfg.attn_implementation = "eager"

In [17]:
calib_tokens = [
    tok(text, return_tensor="pt", padding="max_length", truncation=True, max_length=128).input_ids
    for text in calib_texts[:50]
]

In [19]:
mdl.quantize(
    tok, 
    quant_config=quant_config,
    calib_data=calib_tokens,
    max_calib_seq_len=128,
    max_calib_samples=50,
    n_parallel_calib_samples=1
)

AWQ: 100%|██████████| 22/22 [05:39<00:00, 15.41s/it]
